# Qwen3-VL-32B-Instruct 적용 버전

베이스라인(Qwen2.5-VL-3B)에서 Qwen3-VL-32B로 교체한 버전입니다.

**변경 사항 요약:**
- 모델: `Qwen2.5-VL-3B` → `Qwen3-VL-32B-Instruct`
- 이미지 해상도: 384px 고정 → 동적 해상도 (256~1280px)
- 학습 데이터: 200개 샘플 → 전체 데이터
- LoRA rank: 8 → 16
- 학습률: 1e-4 → 2e-5
- Epoch: 1 → 3
- compute_dtype: float16 → bfloat16 (A100 최적화)
- 데이터: GitHub 레포에서 클론

A100 GPU (40GB) 환경 / Colab Pro에서 실행하세요.

# 환경 준비

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade
!pip -q install qwen-vl-utils

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# GitHub에서 데이터 가져오기

아래 토큰과 레포 정보를 본인 것으로 변경하세요.

GitHub Token 발급: GitHub → Settings → Developer settings → Personal access tokens → Generate new token (repo 권한 체크)

In [ ]:
# ── 여기만 수정하세요 ──────────────────────────────────
GITHUB_TOKEN = "ghp_여기에_토큰_입력"
GITHUB_USER  = "본인_깃허브_아이디"
REPO_NAME    = "레포지토리_이름"
# ───────────────────────────────────────────────────────

import os

REPO_DIR = f"/content/{REPO_NAME}"

if os.path.exists(REPO_DIR):
    print("이미 클론된 레포가 있습니다. 최신 상태로 업데이트합니다.")
    !cd {REPO_DIR} && git pull
else:
    print("레포 클론 중...")
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}

print("완료:", REPO_DIR)

In [ ]:
# 데이터셋 zip 압축 해제
ZIP_NAME = "2026_15_2_ai_DataSet"
ZIP_PATH = f"{REPO_DIR}/{ZIP_NAME}.zip"

if os.path.exists(ZIP_PATH):
    print(f"압축 해제 중: {ZIP_PATH}")
    !unzip -q "{ZIP_PATH}" -d "/content/"
    print("압축 해제 완료")
else:
    # zip이 레포 안에 없을 경우 — /content 에 직접 올린 경우
    alt_path = f"/content/{ZIP_NAME}.zip"
    if os.path.exists(alt_path):
        print(f"압축 해제 중: {alt_path}")
        !unzip -q "{alt_path}" -d "/content/"
        print("압축 해제 완료")
    else:
        print(f"[주의] zip 파일을 찾지 못했습니다.")
        print(f"  레포 경로 확인: {ZIP_PATH}")
        print(f"  또는 파일 탭에서 직접 업로드 후 재실행하세요.")

# 압축 해제 후 파일 구조 확인
print("\n/content 파일 목록:")
!ls /content/

# 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_ID   = "Qwen/Qwen3-VL-32B-Instruct"
MIN_PIXELS = 256 * 256
MAX_PIXELS = 1280 * 1280
MAX_NEW_TOKENS = 8
EPOCHS = 3
SEED   = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

train_df = train_df.reset_index(drop=True)
print(f"전체 학습 데이터: {len(train_df)}개")
print(f"테스트 데이터:    {len(test_df)}개")

# 모델, Processor

32B 모델 다운로드: 약 60~70GB, 20~40분 소요됩니다.

In [ ]:
# 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # A100은 bfloat16 네이티브 지원
)

# 프로세서 — 동적 해상도 설정
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    trust_remote_code=True,
)

# 사전학습 모델 로드
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 설정 — 32B 모델에 맞게 rank 확대
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 프롬프트 템플릿

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]
            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc

# DataLoader

In [ ]:
split = int(len(train_df) * 0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,
                          collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=2, shuffle=False,
                          collate_fn=DataCollator(processor, True), num_workers=0)

print(f"Train batches: {len(train_loader)}, Valid batches: {len(valid_loader)}")

# Fine-tuning

In [ ]:
from tqdm.auto import tqdm

model = model.to(device)
GRAD_ACCUM = 4

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(num_training_steps * 0.03), num_training_steps
)

# bfloat16은 스케일러 불필요 (A100 한정)
scaler = torch.cuda.amp.GradScaler(enabled=False)

global_step = 0
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss: {val_loss / val_steps:.4f}")

# 모델 저장
SAVE_DIR = "/content/qwen3_vl_32b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)

# GitHub에 결과 Push (선택사항)

학습된 모델 또는 제출 파일을 GitHub에 올릴 경우 실행하세요.

In [ ]:
# submission.csv를 레포에 추가 후 push
import shutil

# 제출 파일을 레포 디렉토리로 복사
shutil.copy("/content/submission.csv", f"{REPO_DIR}/submission.csv")

%cd {REPO_DIR}
!git config user.email "본인이메일@example.com"
!git config user.name "{GITHUB_USER}"
!git add submission.csv
!git commit -m "add submission.csv"
!git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git

print("Push 완료")

# Inference

In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last
    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"


model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id
        )
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    preds.append(extract_choice(output_text))

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")
print(submission.head())

In [ ]:
# 마지막 추론 결과 확인
print(output_text)